In [ ]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

- 2013년이후 전국 아파트분양가격
- 공공데이터 포털에 있는 데이터를 Pandas의 melt, concat, transpose와 같은 reshape 기능을 활용해 분석
- groupby, pivot_table, info, describe, value_counts등을 통한 데이터 요약과 분석
- 이를 통해 전혀 다른 두 데이터의 정제와 병합, 수치형 데이터의 차이를 이해하고 다양한 그래프로 시각화(barplot, lineplot, scatterplot, lmplot, swarmplot, histplot)

# 1. 패키지로드 & 한글설정 & 경고메세지 ignore

In [ ]:
# 패키지 import
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# 시각화의 선명도를 높임
%config InlineBackend.figure_format = "retina"

# 다른 폰트를 기본으로 하려면
from matplotlib.font_manager import FontProperties, fontManager
font_path = r"C:\Users\Admin\Downloads\JMK체.ttf"
fontManager.addfont(font_path)
# 폰트 다운로드 : 네이버, 구글, 눈누, 산들 등에서 ttf 파일 
font_prop = FontProperties(fname=font_path, size=12)
font_name = font_prop.get_name()

# 한글설정
plt.rc('font', family=font_name)
plt.rc('font', size=12)
plt.rc('axes', unicode_minus=False) # 축의 - 깨짐 방지

# warning(경고) 안보이게
import warnings
# warnings.filterwarnings(action='ignore')

# 2. 데이터 다운로드
- https://www.data.go.kr 에서 "아파트 분양 가격"을 검색 후 csv 다운로드

## (1) 2013년9월부터 2015년8월까지 데이터는 df_first 변수에 담아 탐색

In [ ]:
df_first = pd.read_csv('c:/ai/downloads/shareData/분양가격/전국 평균 평당 분양가격(2013년 9월부터 2015년 8월까지).csv',
                      encoding='cp949')
df_first.head(1)

In [ ]:
pd.options.display.max_columns = 22 # 최대 display가능한 열수 수정

In [ ]:
df_first.sample()

In [ ]:
df_first.info()

In [ ]:
df_first.set_index('지역', inplace=True)
df_first

- color map 참조 : https://matplotlib.org/stable/users/explain/colors/colormaps.html

In [ ]:
plt.figure(figsize=(25, 5))
sns.heatmap(df_first, annot=True, fmt='d', cmap='Greens', cbar=False)
plt.xticks(rotation=45)
plt.ylabel(None)
plt.show()

## (2) 이후 데이터는 df_last 변수에 담아 탐색

In [ ]:
df_last = pd.read_csv('c:/ai/downloads/shareData/분양가격/주택도시보증공사_전국 신규 민간아파트 분양가격 동향_20260531 (1).csv',
                     encoding='cp949')
df_last.shape

In [ ]:
df_last.tail(3)

In [ ]:
# 데이터 요약하기
df_last.info()

In [ ]:
df_last['분양가격(제곱미터당 천원)'].describe()

# 3. df_last의 데이터 결측치를 확인하고 대체 고민

In [ ]:
# 열별 결측치 갯수
df_last.isna().sum()

In [ ]:
# 결측치인 행
df_last[df_last['분양가격(제곱미터당 천원)'].isna()]

In [ ]:
# 열별 결측치를 제외한 갯수
df_last.count()

# 4. df_last 데이셋에 평당분양가격컬럼 추가

## (1) astype('float')나 pd.to_numeric() 사용
## (2) 6,222 => 6222 수정 후, 형변환
## (3) df_last['분양가격(제곱미터)']*3.3

In [ ]:
# df_last['분양가격(제곱미터당 천원)'].astype('float64') - '  ' 존재로 astype 에러 발생
pd.to_numeric(df_last['분양가격(제곱미터당 천원)'], errors='coerce')
# 변경 후 결측치 개수 973 = 원본 NaN 845 + ' ','  ',etc 128

In [ ]:
# '분양가격(제곱미터당 천원)' = ' ','  '인 행 126 >>> 128 중 ' ', '  '가 아닌 요소 2개
df_last[df_last['분양가격(제곱미터당 천원)'].str.strip()=='']

In [ ]:
# '분양가격(제곱미터당 천원)'의 요소 종류 파악 (5,000 형식의 데이터 있는지 확인)
temp = pd.to_numeric(df_last['분양가격(제곱미터당 천원)'], errors='coerㅠce')
nanidx = temp[temp.isna()].index
df_last.loc[nanidx, '분양가격(제곱미터당 천원)'].unique()

In [ ]:
# '분양가격(제곱미터당 천원)' = '-'인 행 2
df_last[df_last['분양가격(제곱미터당 천원)'].str.strip()=='-']

### apply

In [ ]:
# 콤마가 있는 문자에서 콤마를 없애는 방법
temp = '5,000'
temp.replace(',','')

In [ ]:
# nan >>> nan , ' ' >>> nan, '-' >>> nan, '숫자' >>> 숫자
def commadrop(row):
    row = row.copy()
    price = row['분양가격(제곱미터당 천원)']
    
    if pd.isna(price):
        row['분양가격(제곱미터당 천원)'] = np.nan
        
    elif price.strip() == '' or price == '-':
        row['분양가격(제곱미터당 천원)'] = np.nan
        
    elif price.find(',') != -1:
        row['분양가격(제곱미터당 천원)'] = float(price.replace(',',''))
        
    else:
        row['분양가격(제곱미터당 천원)'] = float(price)
        
    return row
        
print(commadrop(df_last.loc[0])) # '숫자' >>> 숫자
print(commadrop(df_last.loc[28])) # ' ' >>> nan
print(commadrop(df_last.loc[368])) # nan >>> nan
print(commadrop(df_last.loc[8863])) # '-' >>> nan

In [ ]:
# 공백문자는 결측치로, 콤마 제거하기를 모든 행에 적용
df_last = df_last.apply(commadrop, axis=1)

In [ ]:
df_last['분양가격(제곱미터당 천원)'].isna().sum()

### map

In [ ]:
df_last = pd.read_csv('c:/ai/downloads/shareData/분양가격/주택도시보증공사_전국 신규 민간아파트 분양가격 동향_20260531 (1).csv',
                     encoding='cp949')

In [ ]:
# nan >>> nan , ' ' >>> nan, '-' >>> nan, '숫자' >>> 숫자, '숫,자' >>> 숫자
def commadrop2(price):
    if pd.isna(price):
        return np.nan
        
    elif price.strip() == '' or price == '-':
        return np.nan
        
    elif price.find(',') != -1:
        return float(price.replace(',',''))
        
    else:
        return float(price)
        
print(commadrop2(df_last.loc[0, '분양가격(제곱미터당 천원)'])) # '숫자' >>> 숫자
print(commadrop2(df_last.loc[28, '분양가격(제곱미터당 천원)'])) # ' ' >>> nan
print(commadrop2(df_last.loc[368, '분양가격(제곱미터당 천원)'])) # nan >>> nan
print(commadrop2(df_last.loc[8863, '분양가격(제곱미터당 천원)'])) # '-' >>> nan

- apply(함수, axis) : 1차원, 2차원(axis 지정, axis 기본값 : 0)
- map(함수) : 1차원
- applymap(함수) : 2차원(모든 요소에 함수 적용)

In [ ]:
#df_last['분양가격(제곱미터당 천원)'].apply(commadrop2)
df_last['제곱미터당 분양가격'] = df_last['분양가격(제곱미터당 천원)'].map(commadrop2)
df_last.iloc[8863:8866]

### apply/map 사용 않고 적용하기

In [ ]:
pd.to_numeric(df_last['분양가격(제곱미터당 천원)'].str.replace(',',''), errors='coerce')

In [ ]:
df_last['제곱미터당 분양가격'] = pd.to_numeric(df_last['분양가격(제곱미터당 천원)'].str.replace(',',''), errors='coerce')
df_last.info()

In [ ]:
df_last['평당 분양가격'] = df_last['제곱미터당 분양가격'] * 3.3
df_last.info()

In [ ]:
df_last[['제곱미터당 분양가격', '평당 분양가격']].describe()

In [ ]:
cols = ['제곱미터당 분양가격', '평당 분양가격']
fig, axes = plt.subplots(nrows=2, ncols=len(cols), figsize=(12, 6))

plt.subplots_adjust(hspace=0.6, wspace=0.2)

for i, col in enumerate(cols):
    sns.boxplot(data=df_last, x=col, ax = axes[0, i])
    axes[0,i].set_title(f'{col} - boxPlot')
    sns.histplot(data=df_last, x = col, ax = axes[1, i], color='salmon', kde=True)
    axes[1,i].set_title(f'{col} - histPlot')
    axes[1,i].axvline(df_last[col].mean(), c='b', linestyle='-', label=f'mean={df_last[col].mean():.1f}')
    axes[1,i].axvline(df_last[col].median(), c='g', linestyle='--', label=f'median={df_last[col].median():.1f}')
    axes[1,i].legend()

plt.tight_layout()

In [ ]:
# 왜도(skewness) skew : 분포의 비대칭 정도를 수치로 나타내는 지표
    # 0 : 좌우 대칭 / |skew| < 0.5 : 대칭성 높음
    # skew > 0 : 오른쪽 꼬리 긴 분포 (평균 > 중위수)
    # skew < 0 : 왼쪽 꼬리 긴 분포 (중위수 < 평균)
    
df_last[['제곱미터당 분양가격', '평당 분양가격']].skew()

# 5. df_last 데이터셋에서 전용면적 컬럼을 추가한다
- 전용면적 컬럼, 필요없는 컬럼 처리로 메모리 절약

In [ ]:
df_last['규모구분'].unique()

In [ ]:
# 전용면적 60제곱미터이하 >>> 60m²
# 전용면적 60제곱미터초과 85제곱미터이하 >>> 60m²~85m²
# 전용면적 102제곱미터초과 >>> 102m²~

df_last['전용면적'] = df_last['규모구분'].str.replace('전용면적','').str.replace('이하','').str.replace('제곱미터','m²').str.replace(' ','').str.replace('초과','~ ')

In [ ]:
print(df_last['전용면적'].unique())
print(df_last['규모구분'].unique())

# 6. 메모리 사용량을 줄이기 위해 불필요한 컬럼 제거

In [ ]:
df_last.drop(['규모구분', '분양가격(제곱미터당 천원)', '제곱미터당 분양가격'], axis=1, inplace=True)

In [ ]:
# 열 순서 변경하기

# df_last[['지역명','전용면적','연도','월','평당 분양가격']]
# df_last.reindex(columns=['지역명','전용면적','연도','월','평당 분양가격'])
col = df_last.pop('전용면적')
df_last.insert(1, '전용면적', col)

In [ ]:
df_last.head()

# df_last 저장 위치

In [ ]:
df_last.to_csv('data/df_last.csv', index=False) # 전처리 단계 백업

In [ ]:
df_last = pd.read_csv('data/df_last.csv')

# 7. 데이터 집계
- GroupBy(unstack()함수 이용) vs. pivot_table()
- GroupBy가 시간 소요에서 이득인 부분이 있음
- 결측치 대체 전 데이터의 집계와 시각화

```
- df_last.groupby(["인덱스로 사용할 컬럼명"])["계산할 컬럼명"].연산함수()
- df_last.pivot_table(index=["인덱스로 사용할 컬럼명"],
                    values=["계산할 컬럼명"],
                    aggfunc="연산함수명")
- pd.pivot_table(df_last, index=["인덱스로 사용할 컬럼명"],
                    values=["계산할 컬럼명"],
                    aggfunc="연산함수명")
```

## (1) 지역별 데이터수

In [ ]:
df_last.groupby(['지역명'])['전용면적'].count()
df_last['지역명'].value_counts()

In [ ]:
df_last.groupby('지역명')[['평당 분양가격']].count().sort_values('평당 분양가격', ascending=False).T

In [ ]:
pd.pivot_table(df_last, index='지역명', values='평당 분양가격', aggfunc='count').sort_values('평당 분양가격', ascending=False).T

## (2) 지역별 평당분양가격(평균, 최대값, ... 요약통계량)

In [ ]:
pd.pivot_table(df_last, index='지역명', values='평당 분양가격').sort_values('평당 분양가격', ascending=False).T

In [ ]:
df_last.groupby('지역명')[['평당 분양가격']].max().sort_values('평당 분양가격', ascending=False).T

In [ ]:
df_last.groupby('지역명')[['평당 분양가격']].min().sort_values('평당 분양가격', ascending=False).T

In [ ]:
df_last.groupby('지역명')[['평당 분양가격']].median().T

In [ ]:
# 1차원에 대한 describe는 2차원 데이터로 표현됨
df_last.groupby('지역명')['평당 분양가격'].describe().T

In [ ]:
pd.pivot_table(df_last, index='지역명', values='평당 분양가격', aggfunc='describe').T.round(2)

## (3) 전용면적별 평당분양가격(평균)

In [ ]:
df_last.groupby('전용면적')[['평당 분양가격']].mean().sort_values('평당 분양가격', ascending=False).T

In [ ]:
pd.pivot_table(df_last, index='전용면적', values='평당 분양가격').sort_values('평당 분양가격', ascending=False).T

In [ ]:
df_last.groupby('전용면적')['평당 분양가격'].mean().sort_values(ascending=False)

## (4) 지역별, 전용면적별 평당분양가격(평균)

In [ ]:
# unstack() : 최하위 행 인덱스를 열 헤더로 변경
df_last.groupby(['지역명', '전용면적'])[['평당 분양가격']].mean().sort_values('평당 분양가격', ascending=False).round().unstack()

In [ ]:
# 1차원 데이터에 unstack()을 실행할 경우, 2차원으로 표현
df_last.groupby(['지역명', '전용면적'])['평당 분양가격'].mean().sort_values(ascending=False).unstack().round(1)

In [ ]:
g = pd.pivot_table(df_last, index=['지역명', '전용면적'], values='평당 분양가격').sort_values('평당 분양가격', ascending=False).unstack()
g.columns = g.columns.droplevel(0)
g

In [ ]:
df_last.pivot_table(index='지역명', columns='전용면적', values='평당 분양가격')

## (5) 연도, 지역별 평당분양가격(평균)

In [ ]:
r = pd.pivot_table(df_last, index=['연도', '지역명'], values='평당 분양가격').sort_values('평당 분양가격', ascending=False).unstack().round()
r.columns = r.columns.droplevel(0)
r

In [ ]:
plt.figure(figsize=(20,5))
sns.heatmap(r, annot=True, fmt='.0f', cmap='Blues', annot_kws={'fontsize':12})
plt.show()

- color map 참조 : https://matplotlib.org/stable/users/explain/colors/colormaps.html

# 8. matplotlib으로 시각화(위의 도표)
https://www.research.autodesk.com/publications/same-stats-different-graphs/
## (1) 지역별 결측치를 제외한 데이터수(line, bar)

In [ ]:
plt.rcParams['figure.figsize'] = [15,5]
df_last.groupby('지역명')['평당 분양가격'].count().sort_values().plot(kind='line')
plt.show()

In [ ]:
plt.rcParams['figure.figsize'] = [15,5]
df_last.groupby('지역명')['평당 분양가격'].count().sort_values().plot(kind='line', color='r', linestyle='--', rot=40, title='지역별 데이터 수')
plt.show()

In [ ]:
plt.rcParams['figure.figsize'] = [15,5]
ax = df_last.groupby('지역명')[['평당 분양가격']].count().sort_values('평당 분양가격')
ax.plot(kind='bar', colormap='Set2', rot=0, ylim=(400,700))
plt.title('지역별 데이터 수')
plt.ylabel('데\n이\n터\n수', rotation=0, labelpad=10, va='center')
plt.show()

## (2) 지역별 평당분양가격 (line, bar)

In [ ]:
r = df_last.groupby('지역명')['평당 분양가격'].mean().sort_values()
ax = r.plot(kind='line')
plt.xticks(range(len(r)))
ax.set_xticklabels(r.index, color='b')
plt.show()

In [ ]:
r = df_last.groupby('지역명')['평당 분양가격'].mean().sort_values()
ax = r.plot(kind='bar', rot=0)
plt.xticks(range(len(r)))
ax.set_xticklabels(r.index, color='b')
plt.show()

## (3) 전용면적별 평당분양가격(barplot)

In [ ]:
order=['60m²','60m²~ 85m²','85m²~ 102m²','102m²~ ','모든면적']
r = df_last.groupby('전용면적')['평당 분양가격'].mean().reindex(order)
ax = r.plot(kind='bar', rot=0, ylim=[r.min()-500, r.max()+300])
plt.xticks(range(len(r)))
plt.ylabel('평당\n분양\n가격', rotation=0, labelpad=20, va='center')
ax.set_xticklabels(r.index)
for i, val in enumerate(r):
    plt.text(i-0.1, val+50, round(val), fontsize=15) # 수치를 통한 위치 변경
plt.show()

In [ ]:
order=['60m²','60m²~ 85m²','85m²~ 102m²','102m²~ ','모든면적']
r = df_last.groupby('전용면적')['평당 분양가격'].mean().reindex(order)
ax = r.plot(kind='bar', rot=0, ylim=[r.min()-500, r.max()+300])
plt.xticks(range(len(r)))
plt.ylabel('평당\n분양\n가격', rotation=0, labelpad=20, va='center')
ax.set_xticklabels(r.index)
for i, val in enumerate(r):
    plt.text(i, val, round(val), fontsize=15, va='bottom', ha='center') # va/ha를 사용한 위치 변경
plt.show()

In [ ]:
order=['60m²','60m²~ 85m²','85m²~ 102m²','102m²~ ','모든면적']
r = df_last.groupby('전용면적')['평당 분양가격'].mean().reindex(order)
ax = r.plot(kind='bar', rot=0, ylim=[r.min()-500, r.max()+300])
plt.xticks(range(len(r)))
plt.ylabel('평 당\n분 양\n가 격', rotation=0, labelpad=20, va='center', fontsize=14)
plt.xlabel('전 용 면 적', rotation=0, labelpad=20, ha='center', fontsize=14)
ax.set_xticklabels(r.index)
ax.bar_label(ax.containers[0], fmt='%.0f', padding=3) # bar_label을 이용한 값 표기
plt.show()

## (4) 연도별 평균 평당분양가격(line)

In [ ]:
r = df_last.groupby('연도')['평당 분양가격'].mean()
ax = r.plot(kind='line',marker='o', ylim=[r.min()-500,r.max()+2000])

plt.ylabel('평 당\n분 양\n가 격', rotation=0, labelpad=20, va='center', fontsize=14)
plt.xlabel('연 도', rotation=0, labelpad=20, ha='center', fontsize=14)

for year, val in r.items():
    plt.text(year, val+500, round(val), fontsize=15, va='bottom', ha='center')

plt.show()

범례:https://stackoverflow.com/questions/30490740/move-legend-outside-figure-in-seaborn-tsplot

# 9. seaborn으로 시각화
- 위에서 시각화한 내용을 seaborn으로 시각화

In [ ]:
sns.set(style='white', rc={'figure.figsize':(9,3)})
plt.rc('font', family='Malgun Gothic') # 윈도우즈
plt.rc('axes', unicode_minus=False) # 축의 - 깨짐 방지

## (1) 지역별 평당분양가격

In [ ]:
sns.barplot(data=df_last, x='지역명', y='평당 분양가격', errorbar=None, )
plt.show()

In [ ]:
r = df_last.groupby('지역명')['평당 분양가격'].mean().sort_values(ascending=False)
sns.barplot(data=df_last, x='지역명', y='평당 분양가격', errorbar=None, order=(r.index), palette='cool')
plt.show()

In [ ]:
r = df_last.groupby('지역명')['평당 분양가격'].mean().sort_values(ascending=False)
ax = sns.barplot(data=df_last, x='지역명', y='평당 분양가격', errorbar=None, order=(r.index), palette='cool')

for i, val in enumerate(r):
    plt.text(i, val, round(val), va='bottom', ha='center')

ax.set_ylim([5000, 35000])
plt.show()

## (2) 연도별 평당분양가격

In [ ]:
r = df_last.groupby('연도')['평당 분양가격'].mean().sort_values()
ax = sns.barplot(data=df_last, x='연도', y='평당 분양가격', errorbar=None, order=(r.index), palette='cool')

for i, val in enumerate(r):
    plt.text(i, val, round(val), va='bottom', ha='center')

ax.set_ylim([5000, 25000])
plt.show()

## (3) 그 외 lineplot, relplot, boxplot, violinplot, boxexplot, regplot, swarmplot

In [ ]:
r = df_last.groupby('연도')['평당 분양가격'].mean().sort_values()
ax = sns.lineplot(data=df_last, x='연도', y='평당 분양가격')
plt.xticks(range(2015,2027))
ax.set_ylim([5000, 25000])
plt.show()

In [ ]:
r = df_last.groupby('연도')['평당 분양가격'].mean().sort_values()
ax = sns.relplot(data=df_last, x='연도', y='평당 분양가격', kind='line', hue='지역명', col='지역명',  col_wrap=5, facet_kws={'sharex':False})
plt.xticks(range(2015,2027))
plt.tight_layout()
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
plt.show()

In [ ]:
r = df_last.groupby('연도')['평당 분양가격'].mean().sort_values()
ax = sns.catplot(data=df_last, x='연도', y='평당 분양가격', kind='boxen')
ax.fig.set_size_inches(15,5)
plt.tight_layout()
plt.show()

In [ ]:
%%time
# df_last.sample(frac=0.05) - fraction 비율
plt.figure(figsize=(15, 5))
sns.swarmplot(data=df_last.sample(frac=0.5), x='연도', y='평당 분양가격', size=1, hue='연도', palette='turbo')

https://stackoverflow.com/questions/30490740/move-legend-outside-figure-in-seaborn-tsplot : 범례사용(교안 20page)

# 10. 구조가 다른 df_first와 df_last를 연결 후 분석
## (1) concat

```
- new_first :df_first를 [지역명, 연도, 월, 평당분양가격]으로 변환
- new_last : df_last를 [지역명, 연도, 월, 평당분양가격]으로 변환
                (전용면적이 모든면적인 행만 추출후, 전용면적 열을 삭제)
```

In [ ]:
new_first = df_first.melt(id_vars='지역',
                         var_name='연도월',
                         value_name='평당분양가격')
new_first.head()

In [ ]:
new_first['연도'] = new_first['연도월'].map(lambda datestr : int(datestr[:4]))
new_first['월'] = new_first['연도월'].map(lambda datestr : int(datestr[5:-1]))
new_first.head()

In [ ]:
new_first.rename(columns={'지역':'지역명'}, inplace = True)

In [ ]:
new_first = new_first.drop('연도월', axis=1)
new_first.head()

In [ ]:
new_first.rename(columns={'평당분양가격':'평당 분양가격'}, inplace = True)
new_first.head()

In [ ]:
new_last = df_last[df_last['전용면적']=='모든면적']
new_last = new_last.drop('전용면적', axis=1)
new_last.head(1)

In [ ]:
df_all = pd.concat([new_first,new_last]).sort_values(by=['연도', '월']).reset_index(drop=True)

In [ ]:
df_all.iloc[::500]

In [ ]:
df_all.shape

In [ ]:
df_all.to_csv('data/df_all.csv', index=False) # 전처리 단계 백업

In [ ]:
df_all = pd.read_csv('data/df_all.csv')

## (2) 결측치 대체(연도별, 지역별 중위값)

In [ ]:
df_all.isna().sum()

In [ ]:
fillnadf = df_all.groupby(['연도', '지역명'])['평당 분양가격'].median().unstack()
fillnadf

In [ ]:
def fill_median(row):
    row = row.copy()
    if pd.isna(row['평당 분양가격']):
        year = row['연도']
        locate = row['지역명']
        row['평당 분양가격'] = fillnadf.loc[year, locate]
    return row

fill_median(df_all.loc[1010])

In [ ]:
df = df_all.apply(fill_median, axis=1)
df.isna().sum()

## (3) df.csv파일로 출력

In [ ]:
df.to_csv('data/df(최종).csv', index=False) # 전처리 단계 백업

In [ ]:
df = pd.read_csv('data/df(최종).csv')

## (4) 7절부터 9절까지 분석을 한다(전용면적별 분석은 불가)

- 지역명별 데이터 갯수는 결측치가 없어서 동일(groupby나 pivot_table등으로 확인)
- 지역명별 평당분양가격(groupby, pivot_table, plot, seaborn 연습)
- 연도별 지역명별 평당분양가격(groupby, pivot_table, plot, seaborn 연습)

### ① 지역명별 데이터 행 갯수 : 결과가 1차원 시리즈

### ② 지역별 평당분양가격(평균, 최대값, ... 요약통계량)

In [ ]:
# 지역명당 평당분양가격 시각화(lineplot과 matplotlib의 bar, sns의 barplot)

### ③ 연도별 평당분양가격(평균)

In [ ]:
# 연도별 평당분양가격 시각화(lineplot과 matplotlib의 bar, sns의 barplot)

### ④ 연도, 지역별 평당분양가격(평균)

In [ ]:
# 연도별, 지역별 평당분양가격 시각화(heatmap)

In [ ]:
# 년도별 평균 평당분양가격의 추이

In [ ]:
# 위의 그래프가 겹쳐서 작 보이지 않아 분리

In [ ]:
# 관계형 그래프의 서브플롯을 그리는 함수는 relplot
# 범주형 그래프의 서브플롯을 그리는 함수는 catplot

In [ ]:
# 연도별 평당분양가격의 사분위수


In [ ]:
# 연도별 평당분양가격의 사분위수(분포포함 boxenplot)


In [ ]:
# 연도별 평당분양가격의 사분위수(분포포함 violinplot)


In [ ]:
# 연도에 따른 평당분양가격을 회귀식으로 시각화


In [ ]:
# swarmplot은 많은 데이터를 그리기에는 시간이 많이 걸려 5%만 sampling후 그릴 것을 추천
# 연도별 평당분양가격 산점도
 

In [ ]:
# 분양가격을 hist
